# SHS: log income vs log consumption

Load post-redesign **Survey of Household Spending** (Tier A: 2017 / 2019 / 2023)
and plot household **log after-tax income** against **log total consumption**.

Dollars are **CPI-deflated to 2019 CAD** via `load_shs_tier_a(price_base_year=2019)`.

Rather than a single OLS fit, show:

- **weighted** density of the cloud (hexbin of survey weights)
- **weighted conditional quantiles** of log consumption given log income,
  with light smoothing via cubic B-splines

Do not splice with Tier B (1997–2009).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import make_interp_spline

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.survey import load_local_paths, load_shs_tier_a, wquantile

paths = load_local_paths()
for k in ("shs_ry2023", "shs_edm"):
    print(f"{k}: {paths[k]}  exists={paths[k].exists()}")

out = ROOT / "output" / "survey"
out.mkdir(parents=True, exist_ok=True)
print("output ->", out)

## Load Tier A (2019 CAD) and build log measures

- Income: `inc_net` = after-tax household income (`HHTOTINC - TX010`, with fallback)
- Consumption: `consumption` = `TC001` total consumption expenditure
- Keep rows with strictly positive income and consumption for logs
- Confirm CPI deflation to the 2019 price base

In [ ]:
PRICE_BASE_YEAR = 2019
shs = load_shs_tier_a(price_base_year=PRICE_BASE_YEAR)
print("years", sorted(shs["year"].unique().tolist()), "n=", len(shs))
print("price_base_year", shs.attrs.get("price_base_year"), "dollars", shs.attrs.get("dollars"))
assert int(shs.attrs.get("price_base_year", PRICE_BASE_YEAR)) == PRICE_BASE_YEAR

work = shs.copy()
y = pd.to_numeric(work["inc_net"], errors="coerce")
c = pd.to_numeric(work["consumption"], errors="coerce")
w = pd.to_numeric(work["weight"], errors="coerce")
ok = (y > 0) & (c > 0) & np.isfinite(y) & np.isfinite(c) & np.isfinite(w) & (w > 0)
work = work.loc[ok].copy()
work["log_inc"] = np.log(y.loc[ok].to_numpy(dtype=float))
work["log_cons"] = np.log(c.loc[ok].to_numpy(dtype=float))
work["weight"] = w.loc[ok].to_numpy(dtype=float)

print("estimation sample n=", len(work), "sum(weights)=", float(work["weight"].sum()))
display(work.groupby("year")[["log_inc", "log_cons"]].agg(["count", "mean", "std"]))

## Conditional quantiles of log C | log Y

Bin households by log income (equal-count bins), compute **survey-weighted** quantiles of
log consumption in each bin, then smooth the bin midpoints with a cubic spline.

In [ ]:
QUANTILES = (0.10, 0.25, 0.50, 0.75, 0.90)
Q_STYLES = {
    0.10: dict(color="#4C72B0", ls=":", lw=1.6, label="p10"),
    0.25: dict(color="#55A868", ls="--", lw=1.7, label="p25"),
    0.50: dict(color="#C44E52", ls="-", lw=2.4, label="p50"),
    0.75: dict(color="#55A868", ls="--", lw=1.7, label="p75"),
    0.90: dict(color="#4C72B0", ls=":", lw=1.6, label="p90"),
}


def conditional_log_quantiles(
    df: pd.DataFrame,
    *,
    xcol: str = "log_inc",
    ycol: str = "log_cons",
    wcol: str = "weight",
    n_bins: int = 25,
    probs: tuple[float, ...] = QUANTILES,
    min_bin_n: int = 40,
) -> pd.DataFrame:
    """Weighted conditional quantiles of y given x, via equal-count bins of x."""
    x = df[xcol].to_numpy(dtype=float)
    yv = df[ycol].to_numpy(dtype=float)
    wv = df[wcol].to_numpy(dtype=float)
    order = np.argsort(x, kind="mergesort")
    x_s, y_s, w_s = x[order], yv[order], wv[order]
    edges = np.linspace(0, len(x_s), n_bins + 1, dtype=int)
    rows: list[dict] = []
    for b in range(n_bins):
        lo, hi = int(edges[b]), int(edges[b + 1])
        if hi <= lo:
            continue
        xb, yb, wb = x_s[lo:hi], y_s[lo:hi], w_s[lo:hi]
        n = int(len(xb))
        if n < min_bin_n:
            continue
        x_mid = float(np.dot(xb, wb) / wb.sum())
        qvals = np.atleast_1d(wquantile(yb, wb, list(probs)))
        row = {
            "bin": b,
            "n": n,
            "w_sum": float(wb.sum()),
            "x_mid": x_mid,
            "x_min": float(xb.min()),
            "x_max": float(xb.max()),
        }
        for p, qv in zip(probs, qvals):
            row[f"q{int(round(100 * p))}"] = float(qv)
        rows.append(row)
    return pd.DataFrame(rows)


def smooth_curve(x: np.ndarray, y: np.ndarray, *, n_grid: int = 200, k: int = 3):
    """Cubic (or lower) B-spline through sorted unique x; falls back to raw if sparse."""
    order = np.argsort(x)
    xs, ys = np.asarray(x, float)[order], np.asarray(y, float)[order]
    okm = np.isfinite(xs) & np.isfinite(ys)
    xs, ys = xs[okm], ys[okm]
    uniq_x, inv = np.unique(xs, return_inverse=True)
    uniq_y = np.zeros_like(uniq_x, dtype=float)
    counts = np.zeros_like(uniq_x, dtype=float)
    for i, yi in enumerate(ys):
        uniq_y[inv[i]] += yi
        counts[inv[i]] += 1.0
    uniq_y /= counts
    if len(uniq_x) < 4:
        return uniq_x, uniq_y
    kk = min(k, len(uniq_x) - 1)
    spline = make_interp_spline(uniq_x, uniq_y, k=kk)
    xg = np.linspace(uniq_x.min(), uniq_x.max(), n_grid)
    return xg, spline(xg)


def populated_limits(
    df: pd.DataFrame,
    *,
    xcol: str = "log_inc",
    ycol: str = "log_cons",
    wcol: str = "weight",
    q_lo: float = 0.025,
    q_hi: float = 0.975,
    pad: float = 0.12,
) -> tuple[float, float, float, float]:
    """Axis limits from weighted percentiles of the populated cloud."""
    x = df[xcol].to_numpy(dtype=float)
    yv = df[ycol].to_numpy(dtype=float)
    wv = df[wcol].to_numpy(dtype=float)
    x_lo, x_hi = [float(v) for v in np.atleast_1d(wquantile(x, wv, [q_lo, q_hi]))]
    y_lo, y_hi = [float(v) for v in np.atleast_1d(wquantile(yv, wv, [q_lo, q_hi]))]
    return x_lo - pad, x_hi + pad, y_lo - pad, y_hi + pad


q_pooled = conditional_log_quantiles(work)
q_pooled["sample"] = "pooled"
q_by_year = []
for yr, g in work.groupby("year", sort=True):
    qy = conditional_log_quantiles(g, n_bins=20, min_bin_n=25)
    qy["sample"] = str(int(yr))
    q_by_year.append(qy)
q_all = pd.concat([q_pooled] + q_by_year, ignore_index=True)
q_all.to_csv(out / "shs_log_inc_cons_quantiles.csv", index=False)
display(q_pooled.head())
print("Wrote", out / "shs_log_inc_cons_quantiles.csv")

## Figure: pooled Tier A (2017–2023)

Hexbin colour is the **sum of survey weights** in each cell. Quantile curves use
`wquantile` within income bins. Axes zoom to the populated 2.5th–97.5th weighted percentiles.

In [ ]:
TITLE = "Log consumption relative to log income"


def plot_log_inc_cons(
    df: pd.DataFrame,
    qtab: pd.DataFrame,
    *,
    title: str,
    ax=None,
    show_legend: bool = True,
    xlim=None,
    ylim=None,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7.5, 6))
    else:
        fig = ax.figure

    # Weighted density: colour = sum of WEIGHTD in the hex
    hb = ax.hexbin(
        df["log_inc"],
        df["log_cons"],
        C=df["weight"].to_numpy(dtype=float),
        reduce_C_function=np.sum,
        gridsize=45,
        cmap="Greys",
        mincnt=1,
        linewidths=0.0,
        alpha=0.85,
    )

    if xlim is None or ylim is None:
        x0, x1, y0, y1 = populated_limits(df)
        xlim = xlim or (x0, x1)
        ylim = ylim or (y0, y1)

    # C = Y reference within the zoomed window
    diag0 = max(xlim[0], ylim[0])
    diag1 = min(xlim[1], ylim[1])
    if diag1 > diag0:
        ax.plot([diag0, diag1], [diag0, diag1], color="0.65", lw=1.0, ls="-", zorder=1, label="C = Y")

    x_mid = qtab["x_mid"].to_numpy(dtype=float)
    for p in QUANTILES:
        col = f"q{int(round(100 * p))}"
        yg = qtab[col].to_numpy(dtype=float)
        # keep spline support inside the zoomed x-range
        m = (x_mid >= xlim[0]) & (x_mid <= xlim[1])
        xs, ys = smooth_curve(x_mid[m], yg[m])
        sty = Q_STYLES[p]
        ax.plot(xs, ys, zorder=3, **sty)
        ax.plot(
            x_mid[m],
            yg[m],
            color=sty["color"],
            ls="none",
            marker="o",
            ms=2.5,
            alpha=0.35,
            zorder=2,
        )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("log after-tax income")
    ax.set_ylabel("log consumption")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    if show_legend:
        ax.legend(fontsize=8, loc="upper left", framealpha=0.9)
    return fig, ax, hb


xlim = populated_limits(work)[:2]
ylim = populated_limits(work)[2:]
# shared square-ish window helps read the C=Y diagonal
lo = min(xlim[0], ylim[0])
hi = max(xlim[1], ylim[1])
xlim = ylim = (lo, hi)

fig, ax, hb = plot_log_inc_cons(
    work,
    q_pooled,
    title=TITLE,
    xlim=xlim,
    ylim=ylim,
)
cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.03)
cb.set_label("sum of survey weights")
fig.tight_layout()
fig.savefig(out / "shs_log_inc_cons_quantiles.png", dpi=140, bbox_inches="tight")
plt.show()
print("Wrote", out / "shs_log_inc_cons_quantiles.png")
print(f"axis window [{lo:.2f}, {hi:.2f}] (2019 CAD logs)")

## By survey year

Same layout and shared zoom window, one panel per year.

In [ ]:
years = sorted(int(y) for y in work["year"].unique())
fig, axes = plt.subplots(1, len(years), figsize=(4.2 * len(years), 4.6), sharex=True, sharey=True)
if len(years) == 1:
    axes = [axes]

for ax, yr in zip(axes, years):
    g = work[work["year"] == yr]
    qy = q_all[q_all["sample"] == str(yr)]
    plot_log_inc_cons(
        g,
        qy,
        title=str(yr),
        ax=ax,
        show_legend=(yr == years[0]),
        xlim=xlim,
        ylim=ylim,
    )

fig.suptitle(TITLE, y=1.02)
fig.tight_layout()
fig.savefig(out / "shs_log_inc_cons_quantiles_by_year.png", dpi=140, bbox_inches="tight")
plt.show()
print("Wrote", out / "shs_log_inc_cons_quantiles_by_year.png")

## Notes

- Levels are CPI-deflated to **2019 CAD** before taking logs.
- Quantiles use survey weights (`WEIGHTD`) via `wquantile` within equal-count bins of log income.
- Hexbin colour is the **sum of survey weights** in each cell (not unweighted counts).
- Splines smooth bin estimates; markers show the raw bin quantiles.
- Axes zoom to the populated weighted 2.5th–97.5th percentile range.
- The grey diagonal is the C = Y locus in logs.